# Patient-Specific (PS) Upper-Bound Experiments

For each eligible patient (≥5 preictal windows), trains each model on
the **first 60%** of windows (chronological), uses the **next 20%** as
validation (early stopping + Youden threshold), and evaluates on the
**final 20%**.

| Split | Fraction | Purpose |
|-------|----------|---------|
| Train | 60% | Model fitting |
| Val   | 20% | Early stopping + Youden threshold |
| Test  | 20% | Final metrics |

**Output:** `D:/seizure_results/patient_specific_results.json` and
`D:/seizure_results/patient_specific_results.csv`

In [1]:
import os
import sys
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from sklearn.metrics import roc_auc_score
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from scipy.signal import welch
from tqdm import tqdm

sys.path.insert(0, 'D:/Code')
from data_utils import VALID_PATIENTS, DATA_DIR, WIN
from eval_utils import find_youden_threshold, full_evaluate
from models import CNN1D, EEGNet, TCN, EEGConformer

# ── Config ──
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_CHANNELS   = 18
FS           = 256
WEIGHT_DECAY = 1e-4
MAX_EPOCHS   = 100
PATIENCE     = 20
MIN_PRE_WIN  = 5
OUT_DIR      = r'D:\seizure_results'
BANDS        = [(0.5, 4), (4, 8), (8, 13), (13, 30), (30, 40)]

print(f'Device: {DEVICE}')
print(f'Patients: {len(VALID_PATIENTS)}')
print(f'Output dir: {OUT_DIR}')

Device: cuda
Patients: 23
Output dir: D:\seizure_results


## Helper Functions

In [2]:
# Per-model LR and batch size — must match PI training notebooks exactly
MODEL_CFG = {
    '1D-CNN':        {'lr': 1e-4, 'batch_size': 128},  # train_1dcnn.ipynb
    'EEGNet':        {'lr': 1e-3, 'batch_size': 128},  # train_eegnet.ipynb
    'TCN':           {'lr': 1e-3, 'batch_size': 128},  # train_tcn.ipynb
    'EEG-Conformer': {'lr': 1e-4, 'batch_size': 64},   # train_eeg_conformer.ipynb
}


def extract_psd(X):
    """Extract band-power features via Welch PSD."""
    all_feats = []
    for i in range(0, len(X), 500):
        chunk = X[i:i+500]
        freqs, pxx = welch(chunk, fs=FS, axis=-1, nperseg=512)
        feats = [pxx[:, :, (freqs >= lo) & (freqs <= hi)].mean(axis=-1)
                 for lo, hi in BANDS]
        all_feats.append(np.concatenate(feats, axis=1))
    return np.vstack(all_feats)


def make_ps_loaders(X_train, y_train, X_val, y_val, X_test, y_test, batch_size):
    """DataLoaders: weighted sampler for train, sequential for val/test."""
    n0, n1 = int((y_train == 0).sum()), int((y_train == 1).sum())
    if n1 == 0:
        return None, None, None
    w = np.where(y_train == 1, n0 / n1, 1.0).astype(np.float32)
    sampler = WeightedRandomSampler(torch.from_numpy(w), len(w), replacement=True)

    def _dl(X, y, **kw):
        return DataLoader(
            TensorDataset(torch.tensor(X, dtype=torch.float32),
                          torch.tensor(y, dtype=torch.long)),
            batch_size=batch_size, **kw)

    return (_dl(X_train, y_train, sampler=sampler),
            _dl(X_val,   y_val,   shuffle=False),
            _dl(X_test,  y_test,  shuffle=False))


@torch.no_grad()
def collect_probs(model, loader):
    """Return (probs, labels) arrays from a DataLoader."""
    model.eval()
    logits_list, labels_list = [], []
    for x, y in loader:
        logits_list.append(model(x.to(DEVICE)).cpu())
        labels_list.append(y)
    logits = torch.cat(logits_list)
    labels = torch.cat(labels_list).numpy()
    probs  = torch.softmax(logits, dim=1)[:, 1].numpy()
    return probs, labels


def train_ps_dl(model_class, train_loader, val_loader, test_loader, lr):
    """
    Train one DL model on patient-specific data.
    - Early stopping on val AUC (patience=PATIENCE)
    - Youden threshold selected on val set
    - full_evaluate on test set
    """
    model     = model_class().to(DEVICE)
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=5, min_lr=1e-7)
    criterion = nn.CrossEntropyLoss()

    best_val_auc  = -1.0
    best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    patience_left = PATIENCE

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        val_probs, val_labels = collect_probs(model, val_loader)
        try:
            val_auc = roc_auc_score(val_labels, val_probs)
        except ValueError:
            val_auc = 0.5
        scheduler.step(val_auc)

        if val_auc > best_val_auc:
            best_val_auc  = val_auc
            best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_left = PATIENCE
        else:
            patience_left -= 1
            if patience_left == 0:
                break

    model.load_state_dict(best_state)
    model.to(DEVICE)

    val_probs, val_labels = collect_probs(model, val_loader)
    threshold = find_youden_threshold(val_labels, val_probs)

    test_probs, test_labels = collect_probs(model, test_loader)
    return full_evaluate(test_labels, test_probs, threshold, stride_s=300)


def run_ps_for_patient(patient_id, model_classes):
    """Run PS experiments for one patient across all model types."""
    X = np.load(os.path.join(DATA_DIR, f'{patient_id}_X.npy'))
    y = np.load(os.path.join(DATA_DIR, f'{patient_id}_y.npy'))

    n_pre = int((y == 1).sum())
    if n_pre < MIN_PRE_WIN:
        return None, f'only {n_pre} preictal windows'

    n_total   = len(y)
    train_end = int(n_total * 0.6)
    val_end   = int(n_total * 0.8)

    X_train, y_train = X[:train_end],        y[:train_end]
    X_val,   y_val   = X[train_end:val_end], y[train_end:val_end]
    X_test,  y_test  = X[val_end:],          y[val_end:]

    if int((y_val  == 1).sum()) == 0:
        return None, 'no preictal in val split'
    if int((y_test == 1).sum()) == 0:
        return None, 'no preictal in test split'

    results = {'patient': patient_id,
               'n_pre':   n_pre,
               'n_total': n_total}

    # ── PSD + LDA ──
    try:
        lda = LinearDiscriminantAnalysis(solver='svd', priors=[0.5, 0.5])
        lda.fit(extract_psd(X_train), y_train)
        val_prob  = lda.predict_proba(extract_psd(X_val))[:, 1]
        threshold = find_youden_threshold(y_val, val_prob)
        test_prob = lda.predict_proba(extract_psd(X_test))[:, 1]
        results['PSD+LDA'] = full_evaluate(y_test, test_prob, threshold, stride_s=300)
    except Exception as e:
        results['PSD+LDA'] = None
        print(f'    PSD+LDA failed: {e}')

    # ── DL models — each uses its own lr and batch_size ──
    for name, cls in model_classes.items():
        cfg = MODEL_CFG[name]
        train_loader, val_loader, test_loader = make_ps_loaders(
            X_train, y_train, X_val, y_val, X_test, y_test,
            batch_size=cfg['batch_size'])
        if train_loader is None:
            results[name] = None
            continue
        try:
            results[name] = train_ps_dl(
                cls, train_loader, val_loader, test_loader, lr=cfg['lr'])
        except Exception as e:
            results[name] = None
            print(f'    {name} failed: {e}')

    return results, None


MODEL_CLASSES = {
    '1D-CNN':        CNN1D,
    'EEGNet':        EEGNet,
    'TCN':           TCN,
    'EEG-Conformer': EEGConformer,
}
MODEL_NAMES = ['PSD+LDA'] + list(MODEL_CLASSES.keys())
print('Models:', MODEL_NAMES)

Models: ['PSD+LDA', '1D-CNN', 'EEGNet', 'TCN', 'EEG-Conformer']


## Run All Patients

In [3]:
all_ps_results = []
skipped        = []

for pt in tqdm(VALID_PATIENTS, desc='Patients'):
    print(f'\n--- {pt} ---')
    result, reason = run_ps_for_patient(pt, MODEL_CLASSES)
    if result is None:
        print(f'  [skip] {reason}')
        skipped.append((pt, reason))
        continue
    all_ps_results.append(result)

    for mname in MODEL_NAMES:
        m = result.get(mname)
        if isinstance(m, dict):
            print(f'  {mname:20s}: AUC={m["auc"]:.3f}  '
                  f'FAR={m["far"]:.2f}/h  EvtSen={m["event_sensitivity"]:.3f}')
        else:
            print(f'  {mname:20s}: failed / skipped')

print(f'\nDone. {len(all_ps_results)} patients evaluated, {len(skipped)} skipped.')
if skipped:
    for pt, reason in skipped:
        print(f'  skipped {pt}: {reason}')

Patients:   0%|          | 0/23 [00:00<?, ?it/s]


--- chb01 ---


c:\Users\11217\anaconda3\envs\seizure_prediction\lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
Patients:   4%|▍         | 1/23 [01:52<41:16, 112.58s/it]

  PSD+LDA             : AUC=0.770  FAR=6.26/h  EvtSen=1.000
  1D-CNN              : AUC=0.875  FAR=5.12/h  EvtSen=1.000
  EEGNet              : AUC=0.831  FAR=1.32/h  EvtSen=1.000
  TCN                 : AUC=0.528  FAR=8.91/h  EvtSen=1.000
  EEG-Conformer       : AUC=0.872  FAR=4.41/h  EvtSen=1.000

--- chb02 ---


Patients:   9%|▊         | 2/23 [01:53<16:23, 46.83s/it] 

...

  [skip] no preictal in test split

--- chb22 ---


c:\Users\11217\anaconda3\envs\seizure_prediction\lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
Patients:  96%|█████████▌| 22/23 [2:18:25<08:16, 496.77s/it]

  PSD+LDA             : AUC=0.776  FAR=2.27/h  EvtSen=1.000
  1D-CNN              : AUC=0.994  FAR=3.57/h  EvtSen=1.000
  EEGNet              : AUC=0.724  FAR=6.16/h  EvtSen=1.000
  TCN                 : AUC=0.945  FAR=2.92/h  EvtSen=1.000
  EEG-Conformer       : AUC=0.990  FAR=4.86/h  EvtSen=1.000

--- chb23 ---


Patients: 100%|██████████| 23/23 [2:18:25<00:00, 361.13s/it]

  [skip] no preictal in val split

Done. 16 patients evaluated, 7 skipped.
  skipped chb02: no preictal in test split
  skipped chb04: no preictal in test split
  skipped chb05: no preictal in test split
  skipped chb09: no preictal in val split
  skipped chb17: no preictal in test split
  skipped chb21: no preictal in test split
  skipped chb23: no preictal in val split


## Save Results

In [4]:
os.makedirs(OUT_DIR, exist_ok=True)

# ── JSON ──
json_path = os.path.join(OUT_DIR, 'patient_specific_results.json')
with open(json_path, 'w') as f:
    json.dump(all_ps_results, f, indent=2)
print(f'JSON saved: {json_path}')

# ── CSV (flat table) ──
rows = []
for r in all_ps_results:
    row = {'patient': r['patient'],
           'n_pre':   r['n_pre'],
           'n_total': r['n_total']}
    for mname in MODEL_NAMES:
        m = r.get(mname)
        if isinstance(m, dict):
            row[f'{mname}_auc']     = round(m.get('auc',               float('nan')), 4)
            row[f'{mname}_far']     = round(m.get('far',               float('nan')), 4)
            row[f'{mname}_sen']     = round(m.get('sensitivity',       float('nan')), 4)
            row[f'{mname}_evt_sen'] = round(m.get('event_sensitivity', float('nan')), 4)
            row[f'{mname}_f1']      = round(m.get('f1',                float('nan')), 4)
        else:
            for s in ['auc', 'far', 'sen', 'evt_sen', 'f1']:
                row[f'{mname}_{s}'] = float('nan')
    rows.append(row)

df = pd.DataFrame(rows)
csv_path = os.path.join(OUT_DIR, 'patient_specific_results.csv')
df.to_csv(csv_path, index=False)
print(f'CSV saved: {csv_path}')
display(df)

JSON saved: D:\seizure_results\patient_specific_results.json
CSV saved: D:\seizure_results\patient_specific_results.csv


,patient,n_pre,n_total,PSD+LDA_auc,PSD+LDA_far,PSD+LDA_sen,PSD+LDA_evt_sen,PSD+LDA_f1,1D-CNN_auc,1D-CNN_far,...,TCN_auc,TCN_far,TCN_sen,TCN_evt_sen,TCN_f1,EEG-Conformer_auc,EEG-Conformer_far,EEG-Conformer_sen,EEG-Conformer_evt_sen,EEG-Conformer_f1
0,chb01,443,769,0.7704,6.2647,0.9444,1.0,0.3208,0.8746,5.1176,...,0.5278,8.9118,1.0000,1.0,0.2628,0.8721,4.4118,1.0000,1.0,0.4186
1,chb03,153,499,0.8355,8.0000,1.0000,1.0,0.9048,0.3684,10.5000,...,0.1963,12.0000,0.7105,1.0,0.7013,0.6689,7.0000,0.8026,1.0,0.8079
2,chb06,877,1444,0.6482,6.3750,0.7588,1.0,0.8316,0.6663,3.3750,...,0.6902,4.8750,0.8599,1.0,0.9002,0.7246,4.5000,0.8444,1.0,0.8930
3,chb07,296,1025,0.4072,1.0526,0.0270,1.0,0.0510,0.5526,0.4211,...,0.5539,0.2105,0.0000,0.0,0.0000,0.3676,0.0000,0.0000,0.0,0.0000
4,chb08,444,586,0.4989,8.3077,0.5714,1.0,0.6897,0.1729,12.0000,...,0.4535,8.3077,0.6667,1.0,0.7609,0.4974,3.6923,0.3524,1.0,0.5068
5,chb10,847,1304,0.9392,8.0000,0.9725,1.0,0.9783,0.6601,0.0000,...,0.2209,8.0000,0.2902,1.0,0.4444,0.5549,10.0000,0.6745,1.0,0.7963
6,chb11,262,676,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,chb12,229,280,NaN,NaN,0.8571,1.0,0.9231,NaN,NaN,...,NaN,NaN,0.7143,1.0,0.8333,NaN,NaN,0.5179,1.0,0.6824
8,chb13,308,605,0.3012,11.0000,0.6514,1.0,0.7435,0.1865,11.0000,...,0.5115,8.0000,0.7706,1.0,0.8358,0.0994,12.0000,0.7156,1.0,0.7839
9,chb14,590,779,0.4780,1.2500,0.0463,1.0,0.0847,0.4271,9.2500,...,0.6445,1.5000,0.1852,1.0,0.2985,0.5083,9.5000,0.7870,1.0,0.7359


## Summary Statistics

In [5]:
print(f'\n{"="*65}')
print(f'  PS Results Summary  ({len(all_ps_results)} patients evaluated)')
print(f'{"="*65}')
print(f'{"Model":20s}  {"AUC (mean±std)":>20}  {"FAR/h":>8}  {"EvtSen":>8}  n')
print('-' * 65)

for mname in MODEL_NAMES:
    aucs, fars, evts = [], [], []
    for r in all_ps_results:
        m = r.get(mname)
        if not isinstance(m, dict):
            continue
        v = m.get('auc', float('nan'))
        if not np.isnan(v): aucs.append(v)
        v = m.get('far', float('nan'))
        if not np.isnan(v): fars.append(v)
        v = m.get('event_sensitivity', float('nan'))
        if not np.isnan(v): evts.append(v)
    if aucs:
        print(f'{mname:20s}  '
              f'{np.mean(aucs):.4f} +/- {np.std(aucs, ddof=1):.4f}  '
              f'{np.mean(fars):>8.3f}  '
              f'{np.mean(evts):>8.3f}  '
              f'{len(aucs)}')
    else:
        print(f'{mname:20s}  no results')


  PS Results Summary  (16 patients evaluated)
Model                       AUC (mean±std)     FAR/h    EvtSen  n
-----------------------------------------------------------------
PSD+LDA               0.6074 +/- 0.2150     5.361     0.929  10
1D-CNN                0.5517 +/- 0.2697     5.850     0.857  10
EEGNet                0.5678 +/- 0.1796     4.840     0.929  10
TCN                   0.5285 +/- 0.2173     6.454     0.857  10
EEG-Conformer         0.5804 +/- 0.2526     5.597     0.857  10
